# 03 — Baseline model

**Goal:** train the first real win-probability model, a logistic regression, and set the bar every later model must beat.

**Logistic regression** predicts P(Team0 wins) = σ(w · x), where σ(z) = 1 / (1 + e^−z) squashes any number into (0, 1) and x is the game state. Training picks the weights w that make the observed outcomes most likely, which is the same as minimizing **log loss**. It's the natural baseline here: notebook 02 showed win rate vs net worth lead is S-shaped, which is exactly σ's shape.

**How we score models** (`deadlock_predictor.models.evaluate`):
- **Log loss**: the average of −log(probability the model gave to what actually happened). Confident mistakes cost a lot. Always saying 50% scores ln 2 ≈ 0.693.
- **Brier score**: mean squared error of the probability. Always-50% scores 0.25.
- **Calibration**: when the model says 70%, does that team win about 70% of the time? We measure the average gap (expected calibration error, ECE) and plot it.

**Symmetry.** Every input is a Team0 − Team1 diff, or a diff multiplied by something about the whole game (game time, lobby rank), and there is no intercept. Swapping the teams negates every diff, which negates the logit, so P(mirrored) = 1 − P exactly. The model can't learn a side advantage, which is what we want given that train's 50.7% Team0 edge didn't last.

**The test set stays sealed** until the end of Milestone 4. Every number here is on **validation**.

**Before running:** run `python -m deadlock_predictor.models.train_baseline` once (a few minutes).

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from deadlock_predictor.features.snapshots import LABEL
from deadlock_predictor.models.baseline import input_columns, make_model, weight_curves
from deadlock_predictor.models.data import load_split
from deadlock_predictor.models.evaluate import reliability, scores, scores_by_time

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
METRICS = ROOT / "models" / "baseline_metrics.json"
assert METRICS.exists(), "Run `python -m deadlock_predictor.models.train_baseline` first"
metrics = json.loads(METRICS.read_text())
saved = joblib.load(ROOT / "models" / "baseline.joblib")

M3_RUNGS = [1, 2, 3, 4]  # rungs 5-6 (heroes) are Milestone 4
columns = sorted({c for r in M3_RUNGS for c in input_columns(r)})
train = load_split("train", columns)
val = load_split("validation", columns + ["start_time", "duration_s"])
y_val = val[LABEL].to_numpy()
plt.rcParams["figure.figsize"] = (9, 4)
warnings.filterwarnings("ignore", message="The inplace keyword")
print(f"train {len(train):,} rows, validation {len(val):,} rows; best: {metrics['best']}")

## 1. The ladder

Each rung adds one idea, so we can see what each is worth:

| Rung | Inputs |
|---|---|
| 0 | nothing: always 50% |
| 1 | net worth lead, with a weight that varies smoothly with game time |
| 2 | all diffs (stats, objectives, mid-boss; minus `deaths_diff`, which mirrors `kills_diff`), constant weights |
| 3 | all diffs, each with a time-varying weight |
| 4 | rung 3 + net worth lead × lobby rank |

"Time-varying weight" means each diff is multiplied by a few smooth bump functions of `t` (a cubic spline basis), so the model learns a separate, smoothly changing weight for every game time.

In [ ]:
DESCRIBE = {
    "0": "always 50%",
    "1": "net worth × time",
    "2": "all diffs",
    "3": "all diffs × time",
    "4": "+ rank",
}
ladder = pd.DataFrame(
    {r: {"model": DESCRIBE[r], "C": m["C"], **m["validation"]} for r, m in metrics["rungs"].items()}
).T
ladder.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(ladder.model, ladder.log_loss.astype(float))
ax.invert_yaxis()
lo = float(ladder.log_loss.min())
ax.set_xlim(lo - 0.02, 0.70)
for i, v in enumerate(ladder.log_loss.astype(float)):
    ax.text(v + 0.001, i, f"{v:.4f}", va="center")
ax.set(title="Validation log loss by rung (lower is better)", xlabel="log loss");

**Net worth alone does almost all the work.** Rung 1 cuts log loss from 0.693 to 0.560: 92% of the total improvement the ladder achieves. Adding every other diff (rung 2) gains 0.008 more, and letting their weights vary with time (rung 3) another 0.0035. Lobby rank (rung 4) adds nothing (0.00003), so we keep **rung 3**. The training script picks the simplest rung within 0.0005 of the best, because a gain that small is noise, not a reason to keep extra parameters.

The best model picks the eventual winner at 70.2% of snapshots. Accuracy is a blunt measure for probabilities, though: an early 52% prediction counts as "right" or "wrong" just like a late 95% one.

## 2. Regularization barely matters here

`C` is the inverse strength of an L2 penalty that pulls weights towards 0. A penalty protects against **overfitting**: fitting noise in the training data that won't repeat. How much protection a model needs depends on how many parameters it has compared to how much data. Here even the biggest rung has ~100 weights and 2.4M training rows.

In [ ]:
by_C = pd.DataFrame(
    {
        f"rung {r}": {float(C): s["log_loss"] for C, s in m["by_C"].items()}
        for r, m in metrics["rungs"].items()
        if r != "0"
    }
)
by_C.index.name = "C"
by_C.round(5)

Every C gives the same log loss to 4–5 decimals. With 2.4M rows and ≤112 weights there is little room to overfit, so the penalty has nothing to fix. (It would matter with far less data or far more features, like the hero indicators in Milestone 4.)

## 3. Why mirroring is built into the model, not the data

We planned to train on every snapshot plus its mirror (teams swapped). But with this model structure, a mirrored row has **exactly** the same loss as the original: negating x negates w · x, and flipping the label flips which side of σ is scored, so −log σ(w·x) becomes −log(1 − σ(−w·x)), which is the same number. Adding mirrored rows would just count every row twice: the same optimum, at double the cost.

The check: one fitted model, scored on a sample and on the mirrored sample.

In [ ]:
from deadlock_predictor.models.data import mirror

sample = val.sample(100_000, random_state=0)
cols = input_columns(3)
model = saved["model"]
original = scores(sample[LABEL], model.predict_proba(sample[cols])[:, 1])["log_loss"]
flipped = mirror(sample)
mirrored = scores(flipped[LABEL], model.predict_proba(flipped[cols])[:, 1])["log_loss"]
print(f"log loss on sample: {original:.12f}\nlog loss on mirror: {mirrored:.12f}")

## 4. Where in the game each rung helps

Log loss per snapshot time. Early in the game nobody can know much; late, leads are bigger but the selection effect from notebook 01 (games that last long are the ones where the leader failed to close) makes it harder again.

In [ ]:
fitted = {}
for r in M3_RUNGS:
    m = metrics["rungs"][str(r)]
    fitted[r] = make_model(r, m["C"]).fit(train[input_columns(r)], train[LABEL])
preds = {r: fitted[r].predict_proba(val[input_columns(r)])[:, 1] for r in fitted}

fig, ax = plt.subplots()
for r, p in preds.items():
    st = scores_by_time(val.t.to_numpy(), y_val, p)
    ax.plot(st.index / 60, st.log_loss, marker=".", label=f"{r}: {DESCRIBE[str(r)]}")
ax.axhline(np.log(2), color="grey", ls="--", label="always 50%")
ax.set(title="Validation log loss by game time", xlabel="minute", ylabel="log loss")
ax.legend();

- **Early on, nobody knows much:** at 3 minutes log loss is 0.67, barely below a coin flip. Small leads early don't decide games.
- **Predictions are sharpest around 30–35 minutes** (0.37 for rung 3).
- **Late, it gets harder again:** these are the matches where the leader failed to close (notebook 01's selection effect).
- **The extra features pay off late.** Rung 1 (net worth only) and rung 3 are close until ~25 minutes; at 50 minutes rung 3 scores 0.53 against 0.62. Late, the buildings destroyed (barracks, shield generators, Titan) say more than souls do.

## 5. Calibration

The reliability diagram groups predictions into bins (0–10%, 10–20%, ...) and plots the average prediction against how often Team0 actually won. A perfectly calibrated model sits on the diagonal.

In [ ]:
best = metrics["best"]["rung"]
p_best = preds[best]
rel = reliability(y_val, p_best)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], color="grey", ls="--")
ax.plot(rel.predicted, rel.observed, marker="o")
ax.set(
    title=f"Reliability, rung {best} (validation)",
    xlabel="predicted P(Team0 wins)",
    ylabel="observed win rate",
)
rel.round(3)

Close to the diagonal everywhere (ECE 0.0044, i.e. predictions are off by under half a point on average). Above 50%, Team0 wins up to ~1 point more often than predicted: consistent with the small, fading Team0 edge that our symmetric model deliberately ignores.

## 6. What the model learned

For the time-varying rungs, each diff has a weight curve: its effect on the logit per unit of that diff, at each game time. Raw weights aren't comparable across features (a unit of net worth is one soul; a unit of `obj_titan_diff` is a whole Titan), so we also plot each feature's **typical effect**: weight × the feature's standard deviation at that time, i.e. how much a typical lead in that feature moves the logit.

In [ ]:
times = np.arange(180, 3601, 60)
curves = weight_curves(fitted[3], times)
fig, ax = plt.subplots()
ax.plot(times / 60, curves.net_worth_diff * 1000)
ax.set(
    title="Logit change per 1,000 souls of net worth lead (rung 3)",
    xlabel="minute",
    ylabel="Δ logit",
)
ax.axhline(0, color="grey", lw=0.8);

In [ ]:
# Std of each diff at the snapshot times, interpolated to every minute. (`t` is unsigned in
# the data, so convert it to float: pandas' "nearest" lookup miscomputes unsigned differences.)
std = train.groupby("t")[list(curves.columns)].std()
snap_t = std.index.to_numpy(dtype=float)
typical = curves * np.column_stack([np.interp(times, snap_t, std[c]) for c in curves.columns])
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_prop_cycle(color=plt.cm.tab20.colors)  # 15 lines: more colours than the default 10
for c in typical.columns:
    ax.plot(times / 60, typical[c], label=c.removesuffix("_diff"))
ax.axhline(0, color="grey", lw=0.8)
ax.set(
    title="Effect of a typical (1 std) lead on the logit (rung 3)",
    xlabel="minute",
    ylabel="Δ logit",
)
ax.legend(ncol=2, fontsize=8);

What the curves say:
- **A soul is worth less as the game goes on:** +1,000 souls moves the logit by 0.18 at 5 minutes but 0.07 at 30 and 0.01 at 60. Leads grow in absolute size, so the same number of souls is a smaller share.
- **Net worth dominates at every time.** A typical lead's effect peaks around 30 minutes (~2.0 logit).
- **Buildings take over late:** by 40 minutes, typical leads in shield generators (0.50), Titan and Tier 2 (0.28 each) matter more than everything else except net worth.

Read individual weights with care. Many diffs are strongly correlated, so the model can share one effect between several of them. Some weights look backwards for that reason: `level_diff` is *negative* at 25 minutes, and an early Tier 1 kill counts slightly against you (−0.14 at 10 minutes) once net worth is accounted for, probably because the tower's souls are already in `net_worth_diff`. The **predictions** are what we trust; individual weights are a rough picture of what drives them.

## 7. Preview: win-probability curves

The point of the project: run the model on every snapshot of one match. Three random validation matches (the model never trained on them):

In [ ]:
rng = np.random.default_rng(1)
picked = rng.choice(val.match_id.unique(), 3, replace=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, mid in zip(axes, picked, strict=True):
    rows = val.match_id.to_numpy() == mid
    g = val[rows]
    ax.plot(g.t / 60, p_best[rows], marker="o")
    ax.axhline(0.5, color="grey", ls="--")
    winner = "Team0" if g[LABEL].iloc[0] else "Team1"
    ax.set(title=f"match {mid}: {winner} won", xlabel="minute", ylim=(0, 1))
axes[0].set_ylabel("P(Team0 wins)");

## 8. Summary → Milestone 4

| | validation log loss | Brier | ECE |
|---|---|---|---|
| always 50% | 0.6931 | 0.250 | — |
| **baseline (rung 3)** | **0.5489** | **0.188** | **0.004** |

- **The bar to beat is 0.5489** on validation. The test set is still sealed.
- Net worth carries most of the signal; buildings matter late; lobby rank doesn't help; the penalty strength doesn't matter at this data size.
- What a linear model can't do, and Milestone 4 can try: interactions between features (e.g. a net worth lead *with* the Titan down), thresholds, and **heroes** (team composition is the one thing the model knows nothing about).
- The baseline is well calibrated, so a fancier model must beat it on log loss **without** losing calibration.